# VeriGuard | KushalDrishti | SIH26245
**AI-Based Real-Time Monitoring of Training Centres for Attendance and Infrastructure Compliance**

Run the cells top to bottom. Each phase saves a checkpoint to Google Drive, so a crash or a restart only repeats the unfinished phase.

| Phase | What it does | Deck slide it backs |
|---|---|---|
| 0 | Setup, Drive, settings | all |
| 1 | Test video | 3 |
| 2 | Face blur in memory | 3, 4 |
| 3 | Camera health check | 3, 4 |
| 4 | Headcount and equipment detectors | 2, 3 |
| 5 | Attendance vs headcount check | 2, 3 |
| 6 | Small signed data packets, offline buffer, bandwidth | 2, 3 |
| 7 | Accuracy (false positive / false negative) table | 4 |
| 8 | Speed test | 4 |
| 9 | Dashboard | 3 |

**Honest limits.** The attendance claim and the 15 / 30 minute timings are settings. `DEMO_MODE` shortens the timings so a short video can show the full alert logic. Never quote demo numbers as deployment numbers.

## Phase 0: Setup, Drive and settings

In [ ]:
import os, sys, json, time, glob, pickle, sqlite3, hmac, hashlib, subprocess, importlib.util, urllib.request, warnings
from collections import deque
warnings.filterwarnings("ignore")

# Install only what is missing, so Colab's own packages are left alone.
needed = {"ultralytics": "ultralytics", "streamlit": "streamlit", "plotly": "plotly",
          "pyngrok": "pyngrok", "cv2": "opencv-python-headless"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

# Use Google Drive on Colab, a local folder anywhere else.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/VeriGuard_SIH26245"
except ImportError:
    BASE_DIR = os.path.abspath("VeriGuard_SIH26245")

CKPT_DIR, OUT_DIR, DATA_DIR = f"{BASE_DIR}/checkpoints", f"{BASE_DIR}/outputs", f"{BASE_DIR}/dataset"
EVAL_DIR = f"{BASE_DIR}/eval_frames"
for folder in (CKPT_DIR, OUT_DIR, DATA_DIR, EVAL_DIR, f"{OUT_DIR}/evidence_frames"):
    os.makedirs(folder, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Settings (the deck quotes these numbers) -------------------------------
DEMO_MODE = True   # True: short timings so a short video shows the whole alert logic
CFG = dict(
    centre_code="DEMO-CENTRE",
    trade="classroom_demo",          # key of TRADES in Phase 4
    claimed_attendance=18,           # the number the centre filed on the portal
    instructors_in_room=0,           # removed from the count before comparing
    sample_sec=5,                    # centre time per sample: one frame every 5 seconds
    window_samples=180,              # 15 minutes at one frame per 5 seconds
    sustain_samples=360,             # 30 minutes at one frame per 5 seconds
    tolerance=0.20,                  # gap above 20% counts as a mismatch
    video_sample_sec=5,              # how often we take a frame from the test video
    max_samples=300,
    equipment_votes=3,               # item must be seen in most of the last 3 samples
    min_sharpness=15.0, min_brightness=25.0, min_contrast=8.0,   # camera health limits
    raw_video_kbps=1500,             # ASSUMPTION: 720p H.264 stream, used for the bandwidth comparison
    alerts_per_hour=2,               # ASSUMPTION: evidence photos sent per hour
)
if DEMO_MODE:
    CFG.update(video_sample_sec=1, window_samples=6, sustain_samples=12)
    print("DEMO TIMING ON: window 6 samples, sustain 12 samples. Not the deployment timing (180 / 360).")

FORCE_RERUN = set()   # e.g. {"p5_analytics"} to ignore a saved checkpoint

def checkpoint(name, fn):
    """Run fn() once and save the result; later runs load the saved copy."""
    path = f"{CKPT_DIR}/{name}.pkl"
    if os.path.exists(path) and name not in FORCE_RERUN:
        print(f"[checkpoint] restored: {name}")
        with open(path, "rb") as f:
            return pickle.load(f)
    result = fn()
    with open(path, "wb") as f:
        pickle.dump(result, f)
    print(f"[checkpoint] saved: {name}")
    return result

print(f"Device: {DEVICE.upper()} | Workspace: {BASE_DIR}")

## Phase 1: Test video
We try a public classroom video first. If the download fails, upload your own `.mp4` to `dataset/classroom_workshop_test.mp4` and run this cell again. A made-up video is only a plumbing test and is flagged everywhere.

In [ ]:
VIDEO_URL = "https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/classroom.mp4"

def make_synthetic_video(path):
    """Plumbing test only: coloured blocks, not real people, so detectors will count zero."""
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), 20.0, (640, 480))
    for i in range(300):
        frame = np.full((480, 640, 3), 70, dtype=np.uint8)
        cv2.putText(frame, f"Simulated CCTV bay - frame {i}", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        for s in range(5):
            x, y = int(100 + s * 90 + 20 * np.sin((i + s * 10) / 10)), 200 + (s % 2) * 80
            cv2.rectangle(frame, (x, y), (x + 40, y + 90), (0, 200, 100), -1)
            cv2.circle(frame, (x + 20, y - 15), 18, (200, 180, 150), -1)
        out.write(frame)
    out.release()

def acquire_assets():
    path = f"{DATA_DIR}/classroom_workshop_test.mp4"
    synthetic = False
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(VIDEO_URL, path)
        except Exception as err:
            print(f"!! Video download failed ({err}).")
            print("!! Using a SYNTHETIC video. Do not quote any result from it. Upload a real .mp4 instead.")
            make_synthetic_video(path)
            synthetic = True
    cap = cv2.VideoCapture(path)
    ok = cap.isOpened() and cap.get(cv2.CAP_PROP_FRAME_COUNT) > 0
    cap.release()
    if not ok:
        raise RuntimeError(f"Cannot read {path}. Delete it and upload a valid .mp4.")
    return {"video_path": path, "synthetic": synthetic}

ASSETS = checkpoint("p1_assets", acquire_assets)
with open(f"{OUT_DIR}/run_info.json", "w") as f:
    json.dump({"demo_mode": DEMO_MODE, "synthetic_video": ASSETS["synthetic"], "settings": CFG}, f, indent=2)
print("Video ready:", ASSETS["video_path"], "| synthetic:", ASSETS["synthetic"])

## Phase 2: Face blur in memory
Faces are blurred before any counting, saving or sending. The raw frame is never written to disk. The detector is a basic frontal-face finder, so a face turned away may be missed. Say this if a judge asks.

In [ ]:
class FaceAnonymizer:
    """Blur every detected face in RAM. Returns the blurred frame and the number of faces blurred."""
    def __init__(self):
        self.detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

    def anonymize(self, frame_bgr):
        frame = frame_bgr.copy()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = self.detector.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=4, minSize=(24, 24))
        for (x, y, w, h) in faces:
            y1, y2 = max(0, y - 5), min(frame.shape[0], y + h + 5)
            x1, x2 = max(0, x - 5), min(frame.shape[1], x + w + 5)
            roi = frame[y1:y2, x1:x2]
            small = cv2.resize(roi, (max(1, w // 10), max(1, h // 10)), interpolation=cv2.INTER_LINEAR)
            blocky = cv2.resize(small, (x2 - x1, y2 - y1), interpolation=cv2.INTER_NEAREST)
            frame[y1:y2, x1:x2] = cv2.GaussianBlur(blocky, (23, 23), 30)
        return frame, len(faces)

def to_evidence(frame, height=480):
    """Shrink a blurred frame to 480p for the alert photo."""
    h, w = frame.shape[:2]
    return cv2.resize(frame, (int(w * height / h), height), interpolation=cv2.INTER_AREA)

PRIVACY = FaceAnonymizer()
print("Face blur ready.")

## Phase 3: Camera health check
A blocked, dark, blurred or offline camera must not look like an empty room. Those samples are marked `CAMERA_ISSUE` and kept out of the attendance check.

In [ ]:
def check_camera_health(frame):
    """Returns OK, OFFLINE, BLOCKED, TOO_DARK or BLURRED. Limits are in CFG and can be tuned."""
    if frame is None:
        return "OFFLINE"
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if gray.std() < CFG["min_contrast"]:
        return "BLOCKED"
    if gray.mean() < CFG["min_brightness"]:
        return "TOO_DARK"
    if cv2.Laplacian(gray, cv2.CV_64F).var() < CFG["min_sharpness"]:
        return "BLURRED"
    return "OK"

print("Camera health check ready.")

## Phase 4: Headcount and equipment detectors
Headcount uses YOLOv8 with ByteTrack. Equipment uses YOLO-World, which finds items from text names, so no custom training is needed. At one frame every 5 seconds a tracker cannot follow people between frames, so the count is per frame and then averaged over the window.

In [ ]:
TRADES = {   # sanctioned equipment per trade: type the names, no training
    "classroom_demo": ["chair", "laptop", "workbench", "fire extinguisher"],
    "welder": ["welding transformer", "gas cylinder", "protective booth", "fire extinguisher"],
    "tailoring": ["sewing machine", "workbench", "fire extinguisher"],
    "fitter": ["lathe", "workbench", "fire extinguisher"],
}
SAFETY_ITEMS = ["fire extinguisher"]

class HeadcountEngine:
    def __init__(self):
        self.model = YOLO("yolov8n.pt")

    def count(self, frame):
        """Count people with ByteTrack. Keeps no identity, only a number."""
        r = self.model.track(frame, classes=[0], tracker="bytetrack.yaml", persist=True,
                             verbose=False, device=DEVICE)[0]
        return len(r.boxes)

    def count_single(self, frame, device=None):
        """Count on one unrelated frame (used for the accuracy test and the speed test)."""
        r = self.model.predict(frame, classes=[0], verbose=False, device=device or DEVICE)[0]
        return len(r.boxes)

class EquipmentAuditor:
    def __init__(self, items):
        self.items = list(items)
        self.model = YOLO("yolov8s-worldv2.pt")
        self.model.set_classes(self.items)   # set the vocabulary once, not on every frame

    def inspect(self, frame, device=None):
        r = self.model.predict(frame, conf=0.25, verbose=False, device=device or DEVICE)[0]
        counts = {item: 0 for item in self.items}
        shown = frame.copy()
        for box in r.boxes:
            label = self.items[int(box.cls[0])]
            counts[label] += 1
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(shown, (x1, y1), (x2, y2), (255, 120, 0), 2)
            cv2.putText(shown, f"{label} {float(box.conf[0]):.2f}", (x1, max(15, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 120, 0), 2)
        return counts, shown

class EquipmentMemory:
    """PRESENT if seen in most of the last N samples, ABSENT if never seen, otherwise UNVERIFIED."""
    def __init__(self, items, votes):
        self.votes = votes
        self.need = votes // 2 + 1
        self.history = {i: deque(maxlen=votes) for i in items}

    def update(self, counts):
        for item, n in counts.items():
            self.history[item].append(n > 0)

    def status(self):
        out = {}
        for item, h in self.history.items():
            seen = sum(h)
            if len(h) < self.votes:
                out[item] = "UNVERIFIED"
            elif seen >= self.need:
                out[item] = "PRESENT"
            elif seen == 0:
                out[item] = "ABSENT"
            else:
                out[item] = "UNVERIFIED"
        return out

    def unverified(self):
        return {i: "UNVERIFIED" for i in self.history}

SANCTIONED = TRADES[CFG["trade"]]
HEAD = HeadcountEngine()
AUDITOR = EquipmentAuditor(SANCTIONED)
print("Detectors ready. Sanctioned items:", SANCTIONED)

## Phase 5: Attendance vs headcount check
Rules, as written on the slides:
- A sample counts as a mismatch when the gap is above the tolerance (20%).
- It is a **WARNING** while the mismatch is still short, and a **BREACH** only when it lasts the full sustain time (30 minutes).
- Dark, blocked or offline cameras give `CAMERA_ISSUE`, never a breach.

In [ ]:
def judge_sample(claimed, smoothed, breach_run):
    """Pure rule engine. Returns (status, new breach_run, gap_ratio)."""
    gap_ratio = (claimed - smoothed) / claimed
    if gap_ratio > CFG["tolerance"]:
        breach_run += 1
        status = "BREACH_GHOST_ATTENDANCE" if breach_run >= CFG["sustain_samples"] else "WARNING_SUSPECTED_GAP"
    else:
        breach_run, status = 0, "COMPLIANT"
    return status, breach_run, gap_ratio

def run_analytics():
    cap = cv2.VideoCapture(ASSETS["video_path"])
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    step = max(1, int(fps * CFG["video_sample_sec"]))
    claimed = CFG["claimed_attendance"]
    window = deque(maxlen=CFG["window_samples"])
    memory = EquipmentMemory(SANCTIONED, CFG["equipment_votes"])
    rows, breach_run, frame_idx, n = [], 0, 0, 0
    print(f"Reading video: one frame every {CFG['video_sample_sec']} s of video, claimed roll = {claimed}")
    while n < CFG["max_samples"]:
        ok, raw = cap.read()
        if not ok:
            break
        if frame_idx % step == 0:
            n += 1
            row = {"sample_no": n, "centre_time_sec": n * CFG["sample_sec"],
                   "video_time_sec": round(frame_idx / fps, 2), "claimed_attendance": claimed}
            health = check_camera_health(raw)
            row["camera_health"] = health
            if health != "OK":
                row.update(counted_headcount=np.nan, adjusted_headcount=np.nan, smoothed_presence=np.nan,
                           gap_ratio=np.nan, discrepancy_gap=np.nan, compliance_status="CAMERA_ISSUE",
                           breach_run=breach_run, faces_anonymized=0,
                           equipment_status=json.dumps(memory.unverified()), equipment_counts="{}")
            else:
                blurred, faces = PRIVACY.anonymize(raw)          # blur first, then analyse
                heads = HEAD.count(blurred)
                adjusted = max(0, heads - CFG["instructors_in_room"])
                window.append(adjusted)
                smoothed = float(np.mean(window))
                counts, shown = AUDITOR.inspect(blurred)
                memory.update(counts)
                status, breach_run, gap_ratio = judge_sample(claimed, smoothed, breach_run)
                row.update(counted_headcount=heads, adjusted_headcount=adjusted,
                           smoothed_presence=round(smoothed, 2), gap_ratio=round(gap_ratio, 3),
                           discrepancy_gap=round(claimed - smoothed, 2), compliance_status=status,
                           breach_run=breach_run, faces_anonymized=faces,
                           equipment_status=json.dumps(memory.status()), equipment_counts=json.dumps(counts))
                # Evidence photo only on a confirmed breach: the first one, then every 10th sample.
                if status == "BREACH_GHOST_ATTENDANCE" and (breach_run - CFG["sustain_samples"]) % 10 == 0:
                    cv2.imwrite(f"{OUT_DIR}/evidence_frames/breach_sample{n}.jpg", to_evidence(shown),
                                [int(cv2.IMWRITE_JPEG_QUALITY), 65])
            rows.append(row)
        frame_idx += 1
    cap.release()
    if not rows:
        raise RuntimeError("No frames were read from the video.")
    df = pd.DataFrame(rows)
    df.to_csv(f"{OUT_DIR}/telemetry_audit_stream.csv", index=False)
    return df

AUDIT_DF = checkpoint("p5_analytics", run_analytics)
print(AUDIT_DF["compliance_status"].value_counts().to_string())
print(f"{len(AUDIT_DF)} samples saved to outputs/telemetry_audit_stream.csv")

## Phase 6: Small signed packets, offline buffer, bandwidth
One packet per minute goes to the portal. Each packet is signed (HMAC-SHA256) and chained to the one before it, so an edited or deleted log breaks the chain. Packets wait in a local SQLite buffer when the network is down. These packets use the same format and signing as the cloud in the repo, so the cloud accepts them. Set `KD_MASTER_SECRET` in a real deployment.

In [ ]:
MASTER = os.environ.get("KD_MASTER_SECRET", "demo-master-change-me").encode()   # same variable as the cloud and edge code
WORST_FIRST = ["BREACH_GHOST_ATTENDANCE", "WARNING_SUSPECTED_GAP", "CAMERA_ISSUE", "COMPLIANT"]

def canonical(d):
    return json.dumps(d, sort_keys=True, separators=(",", ":"))

def centre_key(code):
    """One key per centre, derived from the master secret (same rule as shared/security.py in the repo)."""
    return hmac.new(MASTER, b"centre:" + code.encode(), hashlib.sha256).digest()

def sign(body, prev_sig):
    return hmac.new(centre_key(body["centre_code"]), (prev_sig + canonical(body)).encode(), hashlib.sha256).hexdigest()

def build_packets(df):
    packets, prev, base_ts = [], "GENESIS", int(time.time())
    for minute, g in df.groupby(df["centre_time_sec"] // 60):
        last = g.iloc[-1]
        status = next(s for s in WORST_FIRST if s in set(g["compliance_status"]))
        body = {"centre_code": CFG["centre_code"], "trade": CFG["trade"], "ts": base_ts + (int(minute) + 1) * 60, "minute": int(minute),
                "claimed_roll": int(last["claimed_attendance"]),
                "mean_headcount": None if g["adjusted_headcount"].isna().all() else round(float(g["adjusted_headcount"].mean()), 1),
                "smoothed_presence": None if pd.isna(last["smoothed_presence"]) else float(last["smoothed_presence"]),
                "status": status, "camera_health": str(last["camera_health"]),
                "equipment": json.loads(last["equipment_status"]), "faces_blurred": True}
        sig = sign(body, prev)
        packets.append({**body, "prev_sig": prev, "sig": sig})
        prev = sig
    return packets

def verify_chain(packets):
    prev = "GENESIS"
    for p in packets:
        body = {k: v for k, v in p.items() if k not in ("prev_sig", "sig")}
        if p["prev_sig"] != prev or p["sig"] != sign(body, prev):
            return False
        prev = p["sig"]
    return True

class Outbox:
    """Store-and-forward buffer: packets are kept until the portal accepts them."""
    def __init__(self, path):
        if os.path.exists(path):
            os.remove(path)
        self.db = sqlite3.connect(path)
        self.db.execute("CREATE TABLE outbox (id INTEGER PRIMARY KEY, payload TEXT, sent INTEGER DEFAULT 0)")

    def add(self, packet):
        self.db.execute("INSERT INTO outbox (payload) VALUES (?)", (canonical(packet),))
        self.db.commit()

    def pending(self):
        return self.db.execute("SELECT COUNT(*) FROM outbox WHERE sent = 0").fetchone()[0]

    def sync(self, online):
        """Send everything pending if online. Returns how many were sent."""
        if not online:
            return 0
        n = self.pending()
        self.db.execute("UPDATE outbox SET sent = 1 WHERE sent = 0")
        self.db.commit()
        return n

def build_network_report():
    packets = build_packets(AUDIT_DF)
    box = Outbox(f"{BASE_DIR}/outbox.db")
    for p in packets:
        box.add(p)
    queued = box.pending()
    sent_when_offline = box.sync(online=False)
    sent_when_back = box.sync(online=True)

    sizes = [len(canonical(p).encode()) for p in packets]
    photos = glob.glob(f"{OUT_DIR}/evidence_frames/*.jpg")
    photo_bytes = float(np.mean([os.path.getsize(f) for f in photos])) if photos else 40 * 1024  # 40 KB if none yet
    raw_per_hour = CFG["raw_video_kbps"] * 1000 / 8 * 3600
    edge_per_hour = float(np.mean(sizes)) * 60 + photo_bytes * CFG["alerts_per_hour"]
    report = {
        "packet_bytes_avg": round(float(np.mean(sizes))),
        "packets_per_hour": 60,
        "edge_kb_per_minute": round(float(np.mean(sizes)) / 1024, 2),
        "raw_stream_mb_per_hour (ASSUMES %d kbps)" % CFG["raw_video_kbps"]: round(raw_per_hour / 1048576, 1),
        "edge_mb_per_hour (ASSUMES %d photos/hour)" % CFG["alerts_per_hour"]: round(edge_per_hour / 1048576, 2),
        "bandwidth_saved_pct": round((1 - edge_per_hour / raw_per_hour) * 100, 2),
        "signature_chain_valid": verify_chain(packets),
        "tamper_test_detected": not verify_chain([{**packets[0], "claimed_roll": 99}] + packets[1:]) if packets else None,
        "offline_test": {"queued": queued, "sent_while_offline": sent_when_offline, "sent_after_reconnect": sent_when_back},
    }
    with open(f"{OUT_DIR}/edge_packet_sample.json", "w") as f:
        json.dump(packets[-1], f, indent=2)
    with open(f"{OUT_DIR}/network_savings_report.json", "w") as f:
        json.dump(report, f, indent=2)
    return report

NETWORK = checkpoint("p6_network", build_network_report)
print(json.dumps(NETWORK, indent=2))

## Phase 7: Accuracy table (false positive / false negative)
This fills the empty table on slide 4. It needs **your own labels**, so no number here is invented.
1. Run the next cell. It saves up to 40 frames to `eval_frames/` and writes `ground_truth.csv`.
2. Open the frames, fill `true_headcount`, and put 1 or 0 for each item in the CSV.
3. Run the cell again to get `accuracy_table.csv`.

Counts are scored frame by frame: a person found in the right place is a true positive, an extra one is a false positive, a missed one is a false negative.

In [ ]:
EVAL_N = 40
GT_PATH = f"{EVAL_DIR}/ground_truth.csv"

def prepare_eval_set():
    cap = cv2.VideoCapture(ASSETS["video_path"])
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    picks = np.linspace(0, max(0, total - 2), min(EVAL_N, total)).astype(int)
    names = []
    for i in picks:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if ok:
            name = f"frame_{int(i):06d}.jpg"
            cv2.imwrite(f"{EVAL_DIR}/{name}", frame)   # the evaluator needs the raw view; keep this folder private
            names.append(name)
    cap.release()
    gt = pd.DataFrame({"frame_file": names, "true_headcount": np.nan})
    for item in SANCTIONED:
        gt[item] = np.nan
    gt.to_csv(GT_PATH, index=False)

def prf(tp, fp, fn):
    return (round(tp / (tp + fp), 3) if tp + fp else None, round(tp / (tp + fn), 3) if tp + fn else None)

def score_counts(pred, true):
    tp = int(np.minimum(pred, true).sum()); fp = int(np.maximum(pred - true, 0).sum()); fn = int(np.maximum(true - pred, 0).sum())
    return (tp, fp, fn)

def score_flags(pred, true):
    pred, true = np.asarray(pred, bool), np.asarray(true, bool)
    return (int((pred & true).sum()), int((pred & ~true).sum()), int((~pred & true).sum()))

def evaluate():
    if not os.path.exists(GT_PATH):
        prepare_eval_set()
        print(f"Created {GT_PATH}. Fill it in, then run this cell again.")
        return None
    gt = pd.read_csv(GT_PATH)
    if gt[["true_headcount", *SANCTIONED]].isna().any().any():
        print("ground_truth.csv still has blanks. Fill true_headcount and every item column (1 or 0), then re-run.")
        return None
    claimed, tol = CFG["claimed_attendance"], CFG["tolerance"]
    pred_heads, pred_items = [], {i: [] for i in SANCTIONED}
    for name in gt["frame_file"]:
        frame = cv2.imread(f"{EVAL_DIR}/{name}")
        blurred, _ = PRIVACY.anonymize(frame)
        pred_heads.append(max(0, HEAD.count_single(blurred) - CFG["instructors_in_room"]))
        counts, _ = AUDITOR.inspect(blurred)
        for i in SANCTIONED:
            pred_items[i].append(counts[i] > 0)
    pred_heads, true_heads = np.array(pred_heads), gt["true_headcount"].to_numpy(float)
    flag = lambda heads: (claimed - heads) / claimed > tol
    other = [i for i in SANCTIONED if i not in SAFETY_ITEMS]
    def pool(items):
        t = np.array([0, 0, 0])
        for i in items:
            t += np.array(score_flags(pred_items[i], gt[i].to_numpy(bool)))
        return tuple(t)
    tasks = {"Classroom Headcount": score_counts(pred_heads, true_heads),
             "Equipment Present": pool(other), "Safety Items": pool([i for i in SANCTIONED if i in SAFETY_ITEMS]),
             "Attendance Mismatch": score_flags(flag(pred_heads), flag(true_heads))}
    rows = []
    for task, (tp, fp, fn) in tasks.items():
        p, r = prf(tp, fp, fn)
        rows.append({"Task": task, "Precision": p, "Recall": r, "False Positive": fp, "False Negative": fn})
    table = pd.DataFrame(rows)
    table.to_csv(f"{OUT_DIR}/accuracy_table.csv", index=False)
    mae = float(np.abs(pred_heads - true_heads).mean())
    print(table.to_string(index=False))
    print(f"Headcount mean absolute error: {mae:.2f} people over {len(gt)} frames")
    return table

ACCURACY = evaluate()

## Phase 8: Speed test
Fills the "CPU load" blank on slide 4. Colab CPUs are not an office PC, so say where it was measured.

In [ ]:
def speed_test(runs=15):
    frame = cv2.imread(sorted(glob.glob(f"{EVAL_DIR}/*.jpg"))[0]) if glob.glob(f"{EVAL_DIR}/*.jpg") else np.full((480, 640, 3), 90, np.uint8)
    def avg_ms(fn):
        fn()   # warm-up
        t0 = time.perf_counter()
        for _ in range(runs):
            fn()
        return round((time.perf_counter() - t0) / runs * 1000, 1)
    cpu = {"headcount_ms": avg_ms(lambda: HEAD.count_single(frame, device="cpu")),
           "equipment_ms": avg_ms(lambda: AUDITOR.inspect(frame, device="cpu"))}
    total = cpu["headcount_ms"] + cpu["equipment_ms"]
    report = {"measured_on": f"CPU of this runtime ({os.cpu_count()} cores), {runs} runs",
              **cpu, "seconds_of_compute_per_5s_sample": round(total / 1000, 2),
              "approx_cpu_busy_pct_at_one_frame_per_5s": round(total / 5000 * 100, 1)}
    with open(f"{OUT_DIR}/speed_report.json", "w") as f:
        json.dump(report, f, indent=2)
    return report

print(json.dumps(speed_test(), indent=2))

## Phase 9: Dashboard
Writes `app.py` and starts it. A public link needs a free ngrok token: add it as a Colab secret named `NGROK_TOKEN`. If you prefer, download `app.py` and the `outputs` folder and run `streamlit run app.py` on your own machine.

In [ ]:
APP_CODE = r'''
import os, glob, json
import pandas as pd
import plotly.express as px
import streamlit as st

OUT = os.environ.get("VG_OUT", "__OUT__")
st.set_page_config(page_title="KushalDrishti | Training Centre Monitor", layout="wide")
st.title("KushalDrishti: Training Centre Compliance Portal")
st.caption("Team VeriGuard | SIH26245 | Ministry of Skill Development and Entrepreneurship")

def load_json(name, default):
    path = os.path.join(OUT, name)
    return json.load(open(path)) if os.path.exists(path) else default

info = load_json("run_info.json", {})
net = load_json("network_savings_report.json", {})
speed = load_json("speed_report.json", {})
pkt = load_json("edge_packet_sample.json", {})
df = pd.read_csv(os.path.join(OUT, "telemetry_audit_stream.csv"))

if info.get("synthetic_video"):
    st.error("SYNTHETIC VIDEO: plumbing test only. Do not quote these results.")
if info.get("demo_mode"):
    st.warning("DEMO TIMING: the warning and breach windows are shortened for a short video.")

last = df.iloc[-1]
c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Centre", info.get("settings", {}).get("centre_code", "DEMO"))
c2.metric("Claimed on portal", int(last["claimed_attendance"]))
counted = last["adjusted_headcount"]
c3.metric("Counted now", "n/a" if pd.isna(counted) else int(counted))
c4.metric("Status", str(last["compliance_status"]).replace("_", " "))
c5.metric("Bandwidth saved", f"{net.get('bandwidth_saved_pct', 'n/a')}%", "vs raw CCTV (assumed)")
st.markdown("---")

t1, t2, t3, t4, t5 = st.tabs(["Attendance check", "Equipment check", "Privacy and evidence", "Low-bandwidth link", "Accuracy and speed"])

with t1:
    st.subheader("Claimed attendance vs counted trainees")
    fig = px.line(df, x="centre_time_sec", y=["claimed_attendance", "smoothed_presence", "adjusted_headcount"],
                  labels={"value": "Trainees", "centre_time_sec": "Centre time (seconds)", "variable": ""})
    fig.update_traces(line=dict(width=3))
    st.plotly_chart(fig, use_container_width=True)
    colours = {"COMPLIANT": "#2E9E5B", "WARNING_SUSPECTED_GAP": "#F59E0B", "BREACH_GHOST_ATTENDANCE": "#E4572E", "CAMERA_ISSUE": "#7A869A"}
    def paint(v):
        return f"background-color: {colours.get(v, '#7A869A')}; color: white; font-weight: bold;"
    show = df.drop(columns=["equipment_counts"])
    styler = show.style
    styler = (styler.map if hasattr(styler, "map") else styler.applymap)(paint, subset=["compliance_status"])
    st.dataframe(styler, use_container_width=True, height=300)

with t2:
    st.subheader("Sanctioned equipment: Present, Absent or Unverified")
    status = json.loads(last["equipment_status"])
    cols = st.columns(max(1, len(status)))
    for col, (item, s) in zip(cols, status.items()):
        col.metric(item.title(), s.title(), delta={"PRESENT": "seen", "ABSENT": "not seen", "UNVERIFIED": "check by visit"}[s],
                   delta_color={"PRESENT": "normal", "ABSENT": "inverse", "UNVERIFIED": "off"}[s])
    st.info("We report what the camera sees. Whether a machine works is not claimed.")

with t3:
    left, right = st.columns(2)
    with left:
        st.markdown("**Detected and used:** anonymous boxes, a headcount per room, equipment status.")
        st.markdown("**Never detected or stored:** faces, biometrics, names, gender, age, raw video.")
        st.markdown("Faces are blurred in memory before anything is counted, saved or sent.")
    with right:
        photos = sorted(glob.glob(os.path.join(OUT, "evidence_frames", "*.jpg")))
        if photos:
            st.image(photos[0], caption="Blurred 480p evidence photo, made only on a confirmed breach")
        else:
            st.success("No confirmed breach in this run, so no evidence photo was made.")

with t4:
    st.subheader("What the centre sends: one small signed packet per minute")
    st.json(pkt)
    st.write({k: v for k, v in net.items()})

with t5:
    acc = os.path.join(OUT, "accuracy_table.csv")
    if os.path.exists(acc):
        st.subheader("False positive / false negative table")
        st.dataframe(pd.read_csv(acc), use_container_width=True)
    else:
        st.info("Accuracy table not created yet. Label the frames in Phase 7 first.")
    st.subheader("Speed")
    st.json(speed)
'''

with open(f"{BASE_DIR}/app.py", "w") as f:
    f.write(APP_CODE.replace("__OUT__", OUT_DIR))

def launch_dashboard(port=8501):
    subprocess.Popen([sys.executable, "-m", "streamlit", "run", f"{BASE_DIR}/app.py", "--server.port", str(port),
                      "--server.headless", "true"], stdout=open("/tmp/streamlit.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(8)
    token = os.environ.get("NGROK_TOKEN")
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("NGROK_TOKEN")
        except Exception:
            token = None
    if not token:
        print("No NGROK_TOKEN found. Add it as a Colab secret, or run `streamlit run app.py` on your own machine.")
        return
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    print("Dashboard link:", ngrok.connect(port).public_url)

print(f"app.py written to {BASE_DIR}/app.py")
launch_dashboard()